In [ ]:
import json
import os

import numpy as np
import polars as pl
from google.colab import drive
from sentence_transformers import SentenceTransformer

# mxbai-embed-large-v1 is Matryoshka-trained — the leading dims pack the most semantic info,
# so truncating from 1024 → 512 retains ~98% of full-dim retrieval quality at half the size.
# At MTEB benchmark it scores ~62 (English avg) which beats MPNet (~58) even after truncation.
EMBEDDING_MODEL = "mixedbread-ai/mxbai-embed-large-v1"
EMBEDDING_VERSION = "v2"  # bump if you change the model again

# Truncate the embedding vector to this many leading dims (Matryoshka). Keep in sync with
# `embed_dim` in mybookrec/settings.py — downstream FeatureSets read the dim from there.
EMBED_DIM = 512

# Store on disk as float16 — halves the Drive footprint (~5 GB → ~2.5 GB at 2.5M books × 512
# dims). Local loader (mybookrec/io/checkpoints.py:load_npy_to_device) upcasts to float32
# automatically. Cosine retrieval quality loss from fp16 storage is <0.1%.
EMBED_STORAGE_DTYPE = np.float16

In [2]:
drive.mount("/content/drive")  # Mount Google Drive to access data files

Mounted at /content/drive


In [ ]:
# Load the combined catalog from Drive — built locally via
# `python -m mybookrec.features.build_embedding_input` (UCSD + silver bulk dedup).
# Schema: book_id (str), title (str), description (str), source (str).
books_df = pl.read_parquet("/content/drive/MyDrive/embedding_input.parquet")

In [4]:
print(len(books_df))

1782579


In [ ]:
# Load embedding model
model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

# fp16 inference — works on T4 for both MPNet and mxbai, halves memory + speeds up encode
# without measurable quality loss for sentence-similarity tasks.
model.half()

# Versioned output paths: separate shard dir + embedding file per model version so re-running
# with a new model doesn't overwrite the previous results.
shard_dir = f"/content/drive/MyDrive/embeddings_{EMBEDDING_VERSION}"
output_path = f"/content/drive/MyDrive/book_embeddings_{EMBEDDING_VERSION}.npy"
os.makedirs(shard_dir, exist_ok=True)

# mxbai-large is comparable to MPNet in throughput on T4. 256 fp16 samples fits comfortably.
BATCH_SIZE = 256
SHARD_SIZE = 50000

# Build the text the encoder sees: prefer "title - description"; fall back to title alone if the
# description is missing (a lot of bulk-imported books do).
books_df = books_df.with_columns(
    pl.when(pl.col("description").is_null() | (pl.col("description") == ""))
    .then(pl.col("title"))
    .otherwise(pl.col("title") + " - " + pl.col("description"))
    .alias("text")
)

num_books = len(books_df)
for shard_idx in range(0, num_books, SHARD_SIZE):
    shard_path = f"{shard_dir}/shard_{shard_idx:08d}.npy"
    if os.path.exists(shard_path):
        print(f"Shard {shard_idx} already exists, skipping...")
        continue
    shard_books = books_df[shard_idx : shard_idx + SHARD_SIZE]
    shard_texts = shard_books["text"].to_list()
    shard_embeddings = model.encode(
        shard_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    # Matryoshka truncate to EMBED_DIM, then re-normalise (truncation breaks unit norm).
    shard_embeddings = shard_embeddings[:, :EMBED_DIM]
    norms = np.linalg.norm(shard_embeddings, axis=1, keepdims=True)
    shard_embeddings = shard_embeddings / np.maximum(norms, 1e-12)
    # Cast to fp16 for disk storage — local loader upcasts back to fp32 on use.
    np.save(shard_path, shard_embeddings.astype(EMBED_STORAGE_DTYPE))
    print(f"Saved shard {shard_idx} ({shard_embeddings.shape}) to {shard_path}")

# Concatenate all shards into a single matrix and save back to Drive.
all_shard = [np.load(f"{shard_dir}/{f}") for f in sorted(os.listdir(shard_dir)) if f.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)
print(f"Final embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")
np.save(output_path, embeddings)
print(f"Saved final embeddings to {output_path}")

In [ ]:
# Save book_id ↔ row-index mapping. This is INDEPENDENT of the embedding model — same row order
# applies to both the granite-384 and MPNet-768 embeddings. Only re-save if you change books_df.
book_id_to_index = {row["book_id"]: idx for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open("/content/drive/MyDrive/book_id_to_index.json", "w") as f:
    json.dump(book_id_to_index, f, indent=2)

# Reverse mapping for index -> book_id (used during inference/eval to look up metadata)
index_to_book_id = {idx: row["book_id"] for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open("/content/drive/MyDrive/index_to_book_id.json", "w") as f:
    json.dump(index_to_book_id, f, indent=2)

In [ ]:
# Optional: round-trip verification — load the embeddings + index from Drive, save to local
# data/transformed/ so you can spot-check shapes and re-use them locally without re-downloading.
# Skip if you're running purely on Colab.

with open("/content/drive/MyDrive/book_id_to_index.json") as f:
    book_id_to_index = json.load(f)
os.makedirs("data/transformed", exist_ok=True)
with open("data/transformed/book_id_to_index.json", "w") as f:
    json.dump(book_id_to_index, f, indent=2)

# Reload + concat shards from Drive, save locally with the versioned name.
all_shard = [np.load(f"{shard_dir}/{f}") for f in sorted(os.listdir(shard_dir)) if f.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)
local_path = f"data/transformed/book_embeddings_{EMBEDDING_VERSION}.npy"
np.save(local_path, embeddings)
print(f"Saved {embeddings.shape} {embeddings.dtype} to {local_path}")